# UOTReg tutorial — embryoid dataset

The full workflow on real single-cell data: estimate the cell-state distribution at any time as a
robust (unbalanced-OT) local-Fréchet barycenter of the observed snapshots, then reconstruct
per-cell trajectories between the estimated distributions.

`SMOKE = 1` runs a fast laptop pass (NOT the paper's numbers); `SMOKE = 0` uses the paper's
settings (a few minutes on CPU, faster on GPU).

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))     # or install the package once: pip install -e ..

from uotreg.config import ModelConfig, TrainConfig, TrajectoryConfig, UOTConfig
from uotreg.data import TensorSampler, samplers_from_arrays
from uotreg.barycenter import DistributionEstimator
from uotreg.trajectory import TrajectoryFitter, FlowMatchingTrajectory
from uotreg.weights import frechet_weights
from uotreg.metrics import mmd_rbf, w2
from uotreg.plotting import plot_trajectories_2d
import uotreg as U

SMOKE  = 1        # 1 = small and fast; 0 = the paper's settings
SAVE   = 0        # 1 = write the fitted trajectories to OUTDIR
DIM    = 20       # number of principal components
DEVICE = "cpu"    # "cuda" / "mps" for real runs
OUTDIR = "../new_results/tutorial"   # written only when SAVE = 1

## Data

Embryoid body differentiation: ~16.8k cells at 5 time points (days 1.5–25.5) in a pooled PCA
space. The method needs exactly two objects: `observed` — a list of `(n_i, d)` arrays, one per
time point, in time order — and their numeric `timepoints`. To use your own data, build these
from any matrix, e.g. `X = adata.obsm["X_pca"][:, :d]` plus a time column of `adata.obs`.

In [ ]:
X = np.load("../data/embryoid/embryoid_pc_20.npy")[:, :DIM]   # (n_cells, d) pooled PC matrix
stage = np.load("../data/embryoid/time_labels.npy")           # (n_cells,) collection stage per cell
stage_to_day = {0: 1.5, 6: 7.5, 12: 13.5, 18: 19.5, 24: 25.5}

timepoints = [stage_to_day[s] for s in sorted(np.unique(stage))]
observed = [X[stage == s] for s in sorted(np.unique(stage))]  # one (n_i, d) cloud per time point
print(f"timepoints {timepoints}; cells per time {[len(a) for a in observed]}; dim {DIM}")

fig, ax = plt.subplots(figsize=(5, 4), dpi=110)
for a, t in zip(observed, timepoints):
    ax.scatter(a[:, 0], a[:, 1], s=4, alpha=0.4, label=f"day {t}")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.legend(fontsize=7); ax.set_title("observed snapshots")
plt.show()

## Step 1 — distribution estimation

The estimate at time `t` is the unbalanced-OT barycenter of the observed snapshots under
nonnegative local-linear kernel weights: a neural generator `G` produces the barycenter, trained
against each contributing snapshot through transport maps `T` and potentials `D`.

Two parameters do the statistical work: the bandwidth `h` (how many neighboring snapshots
contribute — pick it so the effective sample size printed below is roughly 2–4) and `tau` (the
unbalanced tolerance: small `tau` trims mismatched mass, `tau → ∞` recovers balanced OT).
Everything else is ordinary neural-OT training configuration, set below at the paper's values.

In [ ]:
QT = 13.5     # query time: any observed or interior day
H  = 4.0      # bandwidth, checked with the ESS diagnostic below

for h in [2.0, 4.0, 6.0, 8.0]:
    w = frechet_weights(timepoints, QT, h, scheme="positive", threshold=0.01)
    active = ", ".join(f"day {timepoints[i]}: {w.weights[i]:.2f}" for i in w.active_index)
    print(f"h={h:4.1f}  ESS={w.ess:5.2f}   {active}")

### Set up the data samplers and the model

In [ ]:
# standardize the pooled data; the fit runs in standardized coordinates, samples are mapped back
pool = np.concatenate(observed)
mu, sd = pool.mean(0), pool.std(0) + 1e-6
observed_s = [(o - mu) / sd for o in observed]

# minibatch samplers: one per snapshot, plus the pooled cells (used by the generator init)
samplers = samplers_from_arrays(observed_s, device=DEVICE)
pooled_sampler = TensorSampler(np.concatenate(observed_s), device=DEVICE)

model = ModelConfig(dim=DIM, latent_dim=DIM,
                    gen_hidden=64 if SMOKE else 256, gen_layers=4, gen_dropout=0.05,  # generator G
                    map_hidden=64 if SMOKE else 256, map_layers=5,                    # transport maps T
                    pot_hidden=64 if SMOKE else 256, pot_layers=5, dropout=0.05)      # potentials D

train = TrainConfig(outer_iters=15 if SMOKE else 42,   # outer fixed-point rounds
                    d_iters=15 if SMOKE else 50,       # potential updates per round
                    t_iters=5 if SMOKE else 10,        # map updates per potential update
                    g_iters=15 if SMOKE else 50,       # generator updates per round
                    batch_size=32, batch_size_g=64,
                    lr_map=3e-4, lr_pot=3e-4, lr_gen=1e-4,
                    weight_decay_td=1e-10, weight_decay_gen=1e-8,
                    device=DEVICE, verbose=False, seed=0)

uot = UOTConfig(relaxation="one-sided", tau=5.0, divergence="kl")

### Train

In [ ]:
est = DistributionEstimator(model=model, train=train, uot=uot)
est.fit(samplers, timepoints, query_time=QT, h=H, weight_scheme="positive", threshold=0.01,
        init="gaussian", init_data_sampler=pooled_sampler,
        init_kwargs={"iters": 2000 if SMOKE else 10000, "gaussian_scale": 10.0})

### Sample the estimate and check it

In [ ]:
bary = est.sample(2000) * sd + mu             # back to the original PC coordinates
obs_qt = observed[int(np.argmin(np.abs(np.array(timepoints) - QT)))]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4), dpi=110)
gl = np.asarray(est.history["g_loss"])
ax0.plot(np.log10(np.maximum(gl, 1e-12)), lw=1.5)
ax0.set_xlabel("outer iteration"); ax0.set_ylabel(r"$\log_{10}$ G-loss"); ax0.set_title("training loss")
ax1.scatter(pool[:, 0], pool[:, 1], s=3, c="0.9", alpha=0.3)
ax1.scatter(obs_qt[:, 0], obs_qt[:, 1], s=6, c="0.5", alpha=0.35, label=f"observed day {QT}")
ax1.scatter(bary[:, 0], bary[:, 1], s=6, c="tab:purple", alpha=0.4, label="estimate")
ax1.set_xlabel("PC 1"); ax1.set_ylabel("PC 2"); ax1.legend(fontsize=7); ax1.set_title(f"estimate at day {QT}")
fig.tight_layout(); plt.show()

rng = np.random.default_rng(0)
n = min(len(bary), len(obs_qt), 2000)         # metrics expect equal-size samples
a, b = bary[rng.integers(0, len(bary), n)], obs_qt[rng.integers(0, len(obs_qt), n)]
print(f"estimate vs observed at day {QT}:  MMD = {mmd_rbf(a, b):.4f}   W2 = {w2(a, b):.3f}")

## Step 2 — trajectory reconstruction

Estimate the interior days and keep the endpoints raw (they have no neighbor on one side), giving
an anchored series `(raw, est, est, est, raw)`. Then move cells from day 1.5 forward through it
two ways: composed one-sided-UOT maps (`TrajectoryFitter`, the paper's per-cell trajectory), or a
single OT-CFM velocity field integrated as an ODE (`FlowMatchingTrajectory`).

In [ ]:
# `uotreg.estimate` bundles Step 1 (standardize -> samplers -> configs -> fit -> sample) into one
# call; use it for the remaining interior days with the same parameters, reusing the day-13.5 fit.
EST = dict(h=H, tau=5.0, std_mode="std", divergence="kl", relaxation="one-sided",
           gen_hidden=64 if SMOKE else 256, gen_layers=4, map_layers=5,
           d_iters=15 if SMOKE else 50, t_iters=5 if SMOKE else 10, g_iters=15 if SMOKE else 50,
           budget=15 if SMOKE else 42, init_iters=2000 if SMOKE else 10000, gaussian_scale=10.0,
           device=DEVICE)
est_mid = [bary if t == QT else
           U.estimate(observed, timepoints, query_time=t, dim=DIM, n_gen=2000, seed=0, **EST)
           for t in timepoints[1:-1]]

series = [observed[0]] + est_mid + [observed[-1]]     # endpoints raw, interior estimated
X0 = observed[0][:200]                                # the cells to transport from day 1.5

### Fit the two trajectory models

In [ ]:
# UOT maps, warm-started across transitions. NOTE the maps use a LARGER tau (50) than the
# estimator (5): a small tau over-trims the one-sided map and cells overshoot.
traj_cfg = TrajectoryConfig(uot=UOTConfig(relaxation="one-sided", tau=50.0),
                            d_iters=30 if SMOKE else 200, t_iters=10 if SMOKE else 100,
                            warm_start=True, device=DEVICE, verbose=False, seed=0)
uot_fit = TrajectoryFitter(model=ModelConfig(dim=DIM, map_hidden=256, map_layers=5,
                                             pot_hidden=256, pot_layers=5, dropout=0.05),
                           config=traj_cfg).fit(series)
uot_traj = uot_fit.transport(X0)                      # (n_times, n_cells, d)

In [ ]:
# flow matching: one OT-CFM velocity field PER interval (`shared=False` -- where the population
# branches, a single shared field must average the two targets), integrated interval by interval
fm = FlowMatchingTrajectory(dim=DIM, coupling="ot", hidden=256, n_layers=4, shared=False,
                            device=DEVICE, seed=0)
fm.fit(series, times=timepoints, iters=500 if SMOKE else 3000, batch_size=128, verbose=False)
flow_traj = np.asarray(fm.simulate_at_times(X0, n_per=20))
print(f"UOT-map trajectory {uot_traj.shape}; flow trajectory {flow_traj.shape}")

### Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), dpi=110, sharex=True, sharey=True)
for ax, (nm, tr) in zip(axes, [("UOT maps", uot_traj), ("flow matching", flow_traj)]):
    ax.scatter(pool[:, 0], pool[:, 1], s=3, c="0.9", alpha=0.3)
    plot_trajectories_2d(tr, dims=(0, 1), ax=ax, max_cells=60, title=nm)
fig.suptitle("embryoid trajectories, day 1.5 → 25.5")
fig.tight_layout(); plt.show()

if SAVE:
    os.makedirs(OUTDIR, exist_ok=True)
    np.save(os.path.join(OUTDIR, "embryoid_uot_traj.npy"), uot_traj)
    np.save(os.path.join(OUTDIR, "embryoid_flow_traj.npy"), flow_traj)
    print(f"saved trajectories to {OUTDIR}")